# 🔍 Auditoría Completa: Drive → Supabase → R2

Este notebook compara:
1. **HTMLs en Google Drive** (fuente)
2. **Metadata en Supabase** (base de datos)
3. **HTMLs y TXTs en Cloudflare R2** (almacenamiento)

Y te dice qué falta en cada paso.

In [ ]:
# 1. INSTALACIÓN
!pip install supabase google-api-python-client google-auth-httplib2 google-auth-oauthlib boto3 pandas -q
print("✅ Librerías instaladas")

In [ ]:
# 2. IMPORTAR
import os
from google.colab import userdata, files
from supabase import create_client
from google.oauth2 import service_account
from googleapiclient.discovery import build
import boto3
from botocore.client import Config
import pandas as pd
import time
from collections import defaultdict

## 📋 CONFIGURACIÓN

**IMPORTANTE:** Configura estos secretos en Colab (🔑 en el panel izquierdo):

**Supabase:**
- `SUPABASE_URL`
- `SUPABASE_ANON_KEY`

**Cloudflare R2:**
- `R2_ACCOUNT_ID`
- `R2_ACCESS_KEY_ID`
- `R2_SECRET_ACCESS_KEY`
- `R2_BUCKET_NAME`

In [ ]:
# 3. CONEXIÓN A SUPABASE
# IMPORTANTE: En Secrets (🔑), debes crear 2 secretos:
# Nombre: SUPABASE_URL          Valor: https://efxzcglhyosrxtafirhs.supabase.co
# Nombre: SUPABASE_ANON_KEY     Valor: eyJhbGciOiJI...

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_ANON_KEY')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Conexión a Supabase establecida")

In [ ]:
# 4. CONEXIÓN A GOOGLE DRIVE
print("📤 Sube tu archivo de credenciales de Google (service account JSON):")
uploaded = files.upload()

CREDENTIALS_FILE = list(uploaded.keys())[0]
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=SCOPES)
drive_service = build('drive', 'v3', credentials=credentials)
print("✅ Conexión a Google Drive establecida")

In [ ]:
# 5. CONEXIÓN A CLOUDFLARE R2
R2_ACCOUNT_ID = userdata.get('R2_ACCOUNT_ID')
R2_ACCESS_KEY_ID = userdata.get('R2_ACCESS_KEY_ID')
R2_SECRET_ACCESS_KEY = userdata.get('R2_SECRET_ACCESS_KEY')
R2_BUCKET_NAME = userdata.get('R2_BUCKET_NAME', 'penalex-assets')
R2_ENDPOINT = f'https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com'

r2_client = boto3.client(
    's3',
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    config=Config(signature_version='s3v4'),
    region_name='auto'
)
print("✅ Conexión a Cloudflare R2 establecida")

In [ ]:
# 6. DEFINIR CARPETAS DE DRIVE
FOLDERS = {
    'Sala_Constitucional': {
        'id': '1kGbRPySacSvqZITKQ0ll4-T-O3Ns6ieN',
        'tipo': 'sentencia',
        'sala': 'constitucional'
    },
    'Sala_Penal': {
        'id': '1AlySvmxbfCsMV07Rs24FUt6VFQSPgPY1',
        'tipo': 'sentencia',
        'sala': 'penal'
    },
    'Sustanciacion_Constitucional': {
        'id': '1gAXDSFKVAzVCKAXCBVAraKMM7xCXblD8',
        'tipo': 'sentencia',
        'sala': 'sustanciacion'
    },
    'Sala_Penal_Juris': {
        'id': '1BL50vMykxVDiCFIQJuYlhcbvlCoD5afg',
        'tipo': 'jurisprudencia',
        'sala': 'penal'
    },
    'Sala_Constitucional_Juris': {
        'id': '1Ais0FpCjLPOcwOCe8I86hgbGWYj2CQFK',
        'tipo': 'jurisprudencia',
        'sala': 'constitucional'
    },
    'Sala_Plena': {
        'id': '1hgpAKqvge2NF-QrWDv5Lf6EKGfZJzEUo',
        'tipo': 'resoluciones',
        'sala': 'plena'
    }
}

print(f"📁 Carpetas configuradas: {len(FOLDERS)}")

## 📊 PASO 1: Escanear Google Drive (HTMLs)

In [ ]:
def listar_htmls_drive(folder_id, tipo, sala):
    archivos = []
    page_token = None
    query = f"'{folder_id}' in parents and mimeType='text/html' and trashed=false"
    
    while True:
        response = drive_service.files().list(
            q=query,
            fields='nextPageToken, files(id, name)',
            pageToken=page_token,
            pageSize=1000
        ).execute()
        
        for file in response.get('files', []):
            archivos.append({
                'id': file['id'],
                'nombre': file['name'],
                'tipo': tipo,
                'sala': sala
            })
        
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    
    return archivos

print("🔍 PASO 1: Escaneando HTMLs en Google Drive...\n")
print("="*70)

drive_htmls = []
for nombre, config in FOLDERS.items():
    print(f"📂 {nombre}...", end=' ')
    archivos = listar_htmls_drive(config['id'], config['tipo'], config['sala'])
    drive_htmls.extend(archivos)
    print(f"{len(archivos):,} HTMLs")

drive_ids = set([d['id'] for d in drive_htmls])
print(f"\n✅ Total HTMLs en Drive: {len(drive_htmls):,}")

## 📊 PASO 2: Escanear Supabase (Metadata)

In [ ]:
print("\n🔍 PASO 2: Descargando metadata de Supabase...")

supabase_docs = []
offset = 0
batch_size = 1000

while True:
    response = supabase.table('documentos').select('id, tipo, sala, tiene_pdf').range(offset, offset + batch_size - 1).execute()
    if not response.data:
        break
    supabase_docs.extend(response.data)
    print(f"  Descargados: {len(supabase_docs):,}...", end='\r')
    offset += batch_size

df_supabase = pd.DataFrame(supabase_docs)
supabase_ids = set(df_supabase['id'].tolist())
supabase_con_pdf = set(df_supabase[df_supabase['tiene_pdf'] == True]['id'].tolist()) if 'tiene_pdf' in df_supabase.columns else set()

print(f"\n✅ Total documentos en Supabase: {len(df_supabase):,}")
print(f"📄 Documentos marcados con PDF: {len(supabase_con_pdf):,}")

## 📊 PASO 3: Escanear Cloudflare R2 (HTMLs y TXTs)

In [ ]:
print("\n🔍 PASO 3: Escaneando archivos en Cloudflare R2...")

r2_htmls = set()
r2_txts = set()
r2_otros = set()

paginator = r2_client.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=R2_BUCKET_NAME)

total_procesados = 0

for page in pages:
    if 'Contents' in page:
        for obj in page['Contents']:
            key = obj['Key']
            total_procesados += 1
            
            # Extraer ID del nombre del archivo (sin extensión)
            file_id = key.split('/')[-1]
            
            if key.endswith('.html'):
                doc_id = file_id.replace('.html', '')
                r2_htmls.add(doc_id)
            elif key.endswith('.txt'):
                doc_id = file_id.replace('.txt', '')
                r2_txts.add(doc_id)
            else:
                r2_otros.add(file_id)
            
            if total_procesados % 1000 == 0:
                print(f"  Procesados: {total_procesados:,} archivos...", end='\r')

# Documentos que tienen ambos archivos (HTML + TXT)
r2_completos = r2_htmls & r2_txts

print(f"\n✅ Total archivos en R2: {total_procesados:,}")
print(f"   📄 HTMLs: {len(r2_htmls):,}")
print(f"   📝 TXTs:  {len(r2_txts):,}")
print(f"   ✅ Completos (HTML+TXT): {len(r2_completos):,}")
if len(r2_otros) > 0:
    print(f"   📦 Otros archivos: {len(r2_otros):,}")

## 📊 ANÁLISIS COMPLETO

In [ ]:
print("\n" + "="*80)
print("📊 ANÁLISIS COMPLETO: DRIVE → SUPABASE → R2")
print("="*80)

# Calcular diferencias
falta_metadata = drive_ids - supabase_ids  # HTMLs sin metadata
falta_en_r2 = supabase_ids - r2_completos  # Metadata sin archivos en R2
archivos_huerfanos = r2_completos - supabase_ids  # Archivos en R2 sin metadata
metadata_huerfana = supabase_ids - drive_ids  # Metadata sin HTML fuente

# Documentos con solo HTML o solo TXT
solo_html = r2_htmls - r2_txts
solo_txt = r2_txts - r2_htmls

print(f"\n📈 PIPELINE COMPLETO:")
print("="*80)
print(f"""
  Google Drive (HTMLs)      Supabase (Metadata)      Cloudflare R2 (HTML+TXT)
  ════════════════════  →  ═══════════════════  →  ════════════════════════
      {len(drive_htmls):>7,}                 {len(supabase_ids):>7,}                    {len(r2_completos):>7,}
""")

print("\n📋 RESUMEN:")
print("="*80)
print(f"\n1️⃣  DRIVE → SUPABASE (Procesamiento de HTMLs)")
print(f"    ✅ HTMLs procesados:           {len(supabase_ids):>10,}")
print(f"    ❌ HTMLs pendientes:           {len(falta_metadata):>10,}")
print(f"    📊 % Completado:               {(len(supabase_ids)/len(drive_htmls)*100) if len(drive_htmls) > 0 else 0:>9.1f}%")

print(f"\n2️⃣  SUPABASE → R2 (Copia de archivos)")
print(f"    ✅ Completos en R2 (HTML+TXT): {len(r2_completos):>10,}")
print(f"    ❌ Pendientes en R2:           {len(falta_en_r2):>10,}")
print(f"    ⚠️  Solo HTML (sin TXT):       {len(solo_html):>10,}")
print(f"    ⚠️  Solo TXT (sin HTML):       {len(solo_txt):>10,}")
print(f"    📊 % Completado:               {(len(r2_completos)/len(supabase_ids)*100) if len(supabase_ids) > 0 else 0:>9.1f}%")

print(f"\n3️⃣  PIPELINE COMPLETO (DRIVE → R2)")
print(f"    ✅ Documentos completos:       {len(drive_ids & supabase_ids & r2_completos):>10,}")
print(f"    ❌ Documentos incompletos:     {len(drive_ids) - len(drive_ids & supabase_ids & r2_completos):>10,}")
print(f"    📊 % Pipeline completo:        {(len(drive_ids & supabase_ids & r2_completos)/len(drive_htmls)*100) if len(drive_htmls) > 0 else 0:>9.1f}%")

if len(archivos_huerfanos) > 0 or len(metadata_huerfana) > 0:
    print(f"\n⚠️  DATOS HUÉRFANOS (Limpiar?):")
    if len(archivos_huerfanos) > 0:
        print(f"    Archivos R2 sin metadata:      {len(archivos_huerfanos):>10,}")
    if len(metadata_huerfana) > 0:
        print(f"    Metadata sin HTML fuente:      {len(metadata_huerfana):>10,}")

print("\n" + "="*80)

## 📋 DESGLOSE POR CARPETA

In [ ]:
print("\n📂 DESGLOSE POR CARPETA:")
print("="*80)

resultados = []

for nombre, config in FOLDERS.items():
    # Filtrar documentos de esta carpeta
    htmls_carpeta = [d for d in drive_htmls if d['tipo'] == config['tipo'] and d['sala'] == config['sala']]
    ids_carpeta = set([d['id'] for d in htmls_carpeta])
    
    # Contar en cada etapa
    en_supabase = len(ids_carpeta & supabase_ids)
    en_r2 = len(ids_carpeta & r2_completos)
    completos = len(ids_carpeta & supabase_ids & r2_completos)
    
    resultados.append({
        'Carpeta': nombre,
        'HTMLs': len(htmls_carpeta),
        'En Supabase': en_supabase,
        'En R2 (HTML+TXT)': en_r2,
        'Completos': completos,
        '% Completo': f"{(completos/len(htmls_carpeta)*100) if len(htmls_carpeta) > 0 else 0:.1f}%"
    })

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('Completos', ascending=False)
print(df_resultados.to_string(index=False))

## 💾 EXPORTAR LISTAS DE PENDIENTES

In [ ]:
print("\n💾 Generando archivos CSV...\n")

# 1. HTMLs sin procesar
if len(falta_metadata) > 0:
    htmls_pendientes = [d for d in drive_htmls if d['id'] in falta_metadata]
    df_htmls = pd.DataFrame(htmls_pendientes)
    df_htmls.to_csv('1_htmls_sin_procesar.csv', index=False)
    print(f"✅ 1_htmls_sin_procesar.csv ({len(htmls_pendientes):,} registros)")

# 2. Documentos sin archivos en R2
if len(falta_en_r2) > 0:
    docs_sin_r2 = df_supabase[df_supabase['id'].isin(falta_en_r2)]
    docs_sin_r2.to_csv('2_documentos_sin_r2.csv', index=False)
    print(f"✅ 2_documentos_sin_r2.csv ({len(docs_sin_r2):,} registros)")

# 3. Archivos R2 huérfanos (sin metadata)
if len(archivos_huerfanos) > 0:
    df_huerfanos = pd.DataFrame([{'doc_id': aid} for aid in archivos_huerfanos])
    df_huerfanos.to_csv('3_archivos_r2_huerfanos.csv', index=False)
    print(f"✅ 3_archivos_r2_huerfanos.csv ({len(archivos_huerfanos):,} registros)")

# 4. Documentos con solo HTML (sin TXT)
if len(solo_html) > 0:
    df_solo_html = pd.DataFrame([{'doc_id': did} for did in solo_html])
    df_solo_html.to_csv('4_r2_solo_html.csv', index=False)
    print(f"✅ 4_r2_solo_html.csv ({len(solo_html):,} registros)")

# 5. Documentos con solo TXT (sin HTML)
if len(solo_txt) > 0:
    df_solo_txt = pd.DataFrame([{'doc_id': did} for did in solo_txt])
    df_solo_txt.to_csv('5_r2_solo_txt.csv', index=False)
    print(f"✅ 5_r2_solo_txt.csv ({len(solo_txt):,} registros)")

print("\n📥 Descargando archivos...")
if len(falta_metadata) > 0:
    files.download('1_htmls_sin_procesar.csv')
if len(falta_en_r2) > 0:
    files.download('2_documentos_sin_r2.csv')
if len(archivos_huerfanos) > 0:
    files.download('3_archivos_r2_huerfanos.csv')
if len(solo_html) > 0:
    files.download('4_r2_solo_html.csv')
if len(solo_txt) > 0:
    files.download('5_r2_solo_txt.csv')

print("\n✅ Auditoría completa finalizada!")